# Test Scala Data Provenance

### Prerequisites

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

### Import libraries

In [2]:
val packageVersion = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % packageVersion)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

packageVersion: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension


import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension

### Initialize spark session

In [4]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook")
  .withExtensions(new ProvenanceExtension())
  // .config("spark.jars", s"../target/scala-2.13/dp-spark_2.13-$packageVersion.jar")
  // .config("spark.sql.extensions", "org.dataprov.dp.ProvenanceExtension")
  .config("spark.provenance.enabled", "true")
  .getOrCreate()


println(s"Spark provenance enabled: ${spark.conf.get("spark.provenance.enabled")}")
  

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/15 15:54:05 INFO SparkContext: Running Spark version 4.1.1
26/05/15 15:54:05 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/05/15 15:54:05 INFO SparkContext: Java version 17.0.10+7
26/05/15 15:54:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/15 15:54:05 INFO ResourceUtils: ==============================================================
26/05/15 15:54:05 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/15 15:54:05 INFO ResourceUtils: ==============================================================
26/05/15 15:54:05 INFO SparkContext: Submitted application: notebook
26/05/15 15:54:05 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/05/15 15:54:05 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/05/15 15:54:05 INFO SecurityManager: Changing view acls groups to

Spark provenance enabled: true


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@1079aa5

### Create spark dataframe

We create a dataframe with duplicates to test the provenance annotation.
 
The provenance annotation will count the number of duplicates for each tuple.

In [5]:
// val df: DataFrame = spark.createDataFrame(
//     Seq(
//         ("a", "b", "c"),
//         ("a", "b", "c"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("f", "g", "e")
//     )
// ).toDF("A", "B", "C")

// df.show()

val df: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+



df: DataFrame = [A: string, B: string ... 1 more field]

In [6]:
val dfWithProvenance : DataFrame = df.addProvenanceColumn
dfWithProvenance.printSchema()

dfWithProvenance.show()

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: long (nullable = false)

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|              0|
|  d|  b|  e|              1|
|  f|  g|  e|              2|
+---+---+---+---------------+



dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [7]:
def getProvAttr(plan: LogicalPlan, provenanceColName: String): Attribute =
        if (LogicalPlanIntegrity.canGetOutputAttrs(plan)) plan.output.find(_.name == provenanceColName).get else throw new IllegalArgumentException("Plan is not resolved")

getProvAttr(dfWithProvenance.queryExecution.analyzed, "_provenance_tag")

defined function getProvAttr
res7_1: Attribute = AttributeReference(
  name = "_provenance_tag",
  dataType = LongType,
  nullable = false,
  metadata = {}
)

In [8]:
val df2 : DataFrame = df
    .select("A", "B")
    .join(df.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2.show()

val df3: DataFrame = df
  .select("A", "C")
  .join(df.select("B", "C"), "C")
  .select("A", "B", "C")
  .orderBy("A", "B", "C")

val df2WithProvenance : DataFrame = dfWithProvenance
    .select("A", "B")
    .join(dfWithProvenance.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2WithProvenance.show()

val df3WithProvenance: DataFrame = dfWithProvenance
  .select("A", "C")
  .join(dfWithProvenance.select("B", "C"), "C")
  .select("A", "B", "C")
  .orderBy("A", "B", "C")

df3WithProvenance.show()


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|        (0 ⊗ 0)|
|  a|  b|  e|        (0 ⊗ 1)|
|  d|  b|  c|        (1 ⊗ 0)|
|  d|  b|  e|        (1 ⊗ 1)|
|  f|  g|  e|        (2 ⊗ 2)|
+---+---+---+---------------+

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|        (0 ⊗ 0)|
|  d|  b|  e|        (1 ⊗ 1)|
|  d|  g|  e|        (1 ⊗ 2)|
|  f|  b|  e|        (2 ⊗ 1)|
|  f|  g|  e|        (2 ⊗ 2)|
+---+---+---+---------------+



df2: DataFrame = [A: string, B: string ... 1 more field]
df3: DataFrame = [A: string, B: string ... 1 more field]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [9]:
val df4 = df2.union(df3).distinct().orderBy("A", "B", "C")
df4.show()

val df4WithProvenance = df2WithProvenance.union(df3WithProvenance).distinct().orderBy("A", "B", "C")
df4WithProvenance.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|        (0 ⊗ 0)|
|  a|  b|  e|        (0 ⊗ 1)|
|  d|  b|  c|        (1 ⊗ 0)|
|  d|  b|  e|        (1 ⊗ 1)|
|  d|  g|  e|        (1 ⊗ 2)|
|  f|  b|  e|        (2 ⊗ 1)|
|  f|  g|  e|        (2 ⊗ 2)|
+---+---+---+---------------+



df4: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 1 more field]
df4WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

In [10]:
val df5 = df4.select("A", "C").distinct().orderBy("A", "C")
df5.show()

val df5WithProvenance = df4WithProvenance.select("A", "C").distinct().orderBy("A", "C")
df5WithProvenance.show()

+---+---+
|  A|  C|
+---+---+
|  a|  c|
|  a|  e|
|  d|  c|
|  d|  e|
|  f|  e|
+---+---+

+---+---+---------------+
|  A|  C|_provenance_tag|
+---+---+---------------+
|  a|  c|        (0 ⊗ 0)|
|  a|  e|        (0 ⊗ 1)|
|  d|  c|        (1 ⊗ 0)|
|  d|  e|        (1 ⊗ 1)|
|  d|  e|        (1 ⊗ 2)|
|  f|  e|        (2 ⊗ 1)|
|  f|  e|        (2 ⊗ 2)|
+---+---+---------------+



df5: Dataset[org.apache.spark.sql.Row] = [A: string, C: string]
df5WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, C: string ... 1 more field]

Tests for filter and sort

In [11]:
val df6WithProvenance = dfWithProvenance.sort("A", "B", "C").filter("A = 'a' OR A = 'f'  ")
df6WithProvenance.show()


+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|              0|
|  f|  g|  e|              2|
+---+---+---+---------------+



df6WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

Tests for multiple joins

In [12]:
val df7WithProvenance : DataFrame = df2WithProvenance
    .select("A", "B as B1", "C as C1")
    .join(df3WithProvenance.select("A", "B as B2", "C as C2"), "A")
    .select("A", "B1", "C1", "B2", "C2")

df7WithProvenance.show()

+---+-------------------+
|  A|    _provenance_tag|
+---+-------------------+
|  a|((0 ⊗ 1) ⊗ (0 ⊗ 0))|
|  a|((0 ⊗ 0) ⊗ (0 ⊗ 0))|
|  d|((1 ⊗ 1) ⊗ (1 ⊗ 1))|
|  d|((1 ⊗ 1) ⊗ (1 ⊗ 2))|
|  d|((1 ⊗ 0) ⊗ (1 ⊗ 1))|
|  d|((1 ⊗ 0) ⊗ (1 ⊗ 2))|
|  f|((2 ⊗ 2) ⊗ (2 ⊗ 1))|
|  f|((2 ⊗ 2) ⊗ (2 ⊗ 2))|
+---+-------------------+



df7WithProvenance: DataFrame = [A: string, _provenance_tag: string]

In [13]:
val dfDistinct = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("a", "b", "c"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C").addProvenanceColumn

dfDistinct.show()

val test = dfDistinct.select("A", "B", "C").distinct()
test.explain(true)
test.show()

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|              0|
|  a|  b|  c|              1|
|  f|  g|  e|              2|
+---+---+---+---------------+

== Parsed Logical Plan ==
Deduplicate [A#178, B#179, C#180, _provenance_tag#181L]
+- Project [A#178, B#179, C#180, _provenance_tag#181L]
   +- Project [A#178, B#179, C#180, monotonically_increasing_id() AS _provenance_tag#181L]
      +- Project [_1#175 AS A#178, _2#176 AS B#179, _3#177 AS C#180]
         +- LocalRelation [_1#175, _2#176, _3#177]

== Analyzed Logical Plan ==
A: string, B: string, C: string, _provenance_tag: bigint
Deduplicate [A#178, B#179, C#180, _provenance_tag#181L]
+- Project [A#178, B#179, C#180, _provenance_tag#181L]
   +- Project [A#178, B#179, C#180, monotonically_increasing_id() AS _provenance_tag#181L]
      +- Project [_1#175 AS A#178, _2#176 AS B#179, _3#177 AS C#180]
         +- LocalRelation [_1#175, _2#176, _3#177]

== Optimized Logical Plan ==
Agg

dfDistinct: DataFrame = [A: string, B: string ... 2 more fields]
test: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

In [ ]:

import java.sql.Date


val df = spark.createDataFrame(
    Seq(
    ("A", Date.valueOf("2026-01-15"), 10.0, 90),
    ("A", Date.valueOf("2026-01-16"), 10.0, 120),
    ("A", Date.valueOf("2026-01-17"), 5.0, 300),
    ("B", Date.valueOf("2026-01-15"), 100.0, 20),
    ("B", Date.valueOf("2026-01-16"), 100.0, 30),
    ("C", Date.valueOf("2026-01-17"), 80.0, 60),
    ("F", Date.valueOf("2026-01-16"), 50.0, 70)
)).toDF("product", "date", "price", "quantity").addProvenanceColumn

df.show()


+-------+----------+-----+--------+---------------+
|product|      date|price|quantity|_provenance_tag|
+-------+----------+-----+--------+---------------+
|      A|2026-01-15| 10.0|      90|              0|
|      A|2026-01-16| 10.0|     120|              1|
|      A|2026-01-17|  5.0|     300|              2|
|      B|2026-01-15|100.0|      20|              3|
|      B|2026-01-16|100.0|      30|              4|
|      C|2026-01-17| 80.0|      60|              5|
|      F|2026-01-16| 50.0|      70|              6|
+-------+----------+-----+--------+---------------+



import java.sql.Date
df: DataFrame = [product: string, date: date ... 3 more fields]

In [18]:
df.addProvenanceColumn.createOrReplaceTempView("ma_table_test")
val test = spark.sql("SELECT distinct product FROM ma_table_test")

test.show()

val test2 = df.select("product").dropDuplicates("product")
test2.show()

val test3 = df.select("product").distinct()
test3.show()

+-------+---------------+
|product|_provenance_tag|
+-------+---------------+
|      A|    {1 ⊕ 2 ⊕ 0}|
|      B|        {3 ⊕ 4}|
|      C|            {5}|
|      F|            {6}|
+-------+---------------+

+-------+---------------+
|product|_provenance_tag|
+-------+---------------+
|      A|              0|
|      B|              3|
|      C|              5|
|      F|              6|
+-------+---------------+

+-------+---------------+
|product|_provenance_tag|
+-------+---------------+
|      A|              0|
|      A|              1|
|      A|              2|
|      B|              3|
|      B|              4|
|      C|              5|
|      F|              6|
+-------+---------------+



test: DataFrame = [product: string, _provenance_tag: string]
test2: Dataset[org.apache.spark.sql.Row] = [product: string, _provenance_tag: bigint]
test3: Dataset[org.apache.spark.sql.Row] = [product: string, _provenance_tag: bigint]

In [20]:
test.explain(true)

== Parsed Logical Plan ==
'Distinct
+- 'Project ['product]
   +- 'UnresolvedRelation [ma_table_test], [], false

== Analyzed Logical Plan ==
product: string, _provenance_tag: string
Aggregate [product#212], [product#212, concat({, concat_ws( ⊕ , collect_set(cast(_provenance_tag#216L as string), 0, 0)), }) AS _provenance_tag#428]
+- Project [product#212, _provenance_tag#216L]
   +- SubqueryAlias ma_table_test
      +- View (`ma_table_test`, [product#212, date#213, price#214, quantity#215, _provenance_tag#216L])
         +- Project [product#212, date#213, price#214, quantity#215, monotonically_increasing_id() AS _provenance_tag#216L]
            +- Project [_1#208 AS product#212, _2#209 AS date#213, _3#210 AS price#214, _4#211 AS quantity#215]
               +- LocalRelation [_1#208, _2#209, _3#210, _4#211]

== Optimized Logical Plan ==
Aggregate [product#212], [product#212, concat({, concat_ws( ⊕ , collect_set(cast(_provenance_tag#216L as string), 0, 0)), }) AS _provenance_tag#428]
+- L

In [21]:
test2.explain(true)

== Parsed Logical Plan ==
Deduplicate [product#212]
+- Project [product#212, _provenance_tag#216L]
   +- Project [product#212, date#213, price#214, quantity#215, monotonically_increasing_id() AS _provenance_tag#216L]
      +- Project [_1#208 AS product#212, _2#209 AS date#213, _3#210 AS price#214, _4#211 AS quantity#215]
         +- LocalRelation [_1#208, _2#209, _3#210, _4#211]

== Analyzed Logical Plan ==
product: string, _provenance_tag: bigint
Deduplicate [product#212]
+- Project [product#212, _provenance_tag#216L]
   +- Project [product#212, date#213, price#214, quantity#215, monotonically_increasing_id() AS _provenance_tag#216L]
      +- Project [_1#208 AS product#212, _2#209 AS date#213, _3#210 AS price#214, _4#211 AS quantity#215]
         +- LocalRelation [_1#208, _2#209, _3#210, _4#211]

== Optimized Logical Plan ==
Aggregate [product#212], [product#212, first(_provenance_tag#216L, false) AS _provenance_tag#494L]
+- LocalRelation [product#212, _provenance_tag#216L]

== Physic

In [22]:
test3.explain(true)

== Parsed Logical Plan ==
Deduplicate [product#212, _provenance_tag#216L]
+- Project [product#212, _provenance_tag#216L]
   +- Project [product#212, date#213, price#214, quantity#215, monotonically_increasing_id() AS _provenance_tag#216L]
      +- Project [_1#208 AS product#212, _2#209 AS date#213, _3#210 AS price#214, _4#211 AS quantity#215]
         +- LocalRelation [_1#208, _2#209, _3#210, _4#211]

== Analyzed Logical Plan ==
product: string, _provenance_tag: bigint
Deduplicate [product#212, _provenance_tag#216L]
+- Project [product#212, _provenance_tag#216L]
   +- Project [product#212, date#213, price#214, quantity#215, monotonically_increasing_id() AS _provenance_tag#216L]
      +- Project [_1#208 AS product#212, _2#209 AS date#213, _3#210 AS price#214, _4#211 AS quantity#215]
         +- LocalRelation [_1#208, _2#209, _3#210, _4#211]

== Optimized Logical Plan ==
Aggregate [product#212, _provenance_tag#216L], [product#212, _provenance_tag#216L]
+- LocalRelation [product#212, _pro